# 11. Evaluation

This notebook evaluates and compares the semantic retrieval approaches developed in the project.

The evaluation investigates whether incorporating historical user preference information into a natural language product-search query affects retrieval performance. Two retrieval conditions are evaluated using the same evaluation examples, product catalogue, semantic encoder and FAISS index.

The first condition is a query-only semantic retrieval approach, in which the synthetic query is encoded using E5-base-v2 and used to retrieve products from the full Amazon Electronics catalogue. The second condition is the proposed profile-conditioned approach, in which the user's historical preference profile is combined with the synthetic query before being encoded using the same semantic encoder.

Both approaches retrieve the same number of candidate products and are evaluated against held-out user interactions using standard information retrieval metrics. The notebook also examines per-example differences and qualitative retrieval behaviour.

## 11.1 Evaluation Data and Retrieval Results

The evaluation uses the synthetic query-conditioned dataset together with the saved retrieval outputs from the query-only and profile-conditioned semantic retrieval experiments. The retrieval outputs were generated using the same E5-base-v2 semantic encoder and FAISS index over the full Amazon Electronics product catalogue.

Before calculating evaluation metrics, the datasets are checked to ensure that the same evaluation examples and ground-truth products are used for both retrieval conditions.

### Live Demonstration — Evaluation Example

The retrieval task combines a user's historical preference profile with a current natural-language query and evaluates retrieval against a held-out ground-truth product.

In [ ]:
print("USER PREFERENCE PROFILE")
print("-" * 60)

print("Preferred categories:")
for category in [
    "Lightning Cables",
    "Keyboard Skins",
    "Labeling Tapes"
]:
    print(f"  • {category}")

print("\nRelevant previous preference:")
print("  • JSAUX Apple MFi Certified Lightning Cables")

print("\nCURRENT QUERY")
print("-" * 60)
print("lightning cable for iPhone 12")

print("\nGROUND-TRUTH PRODUCT")
print("-" * 60)
print("JSAUX Apple MFi Certified Lightning Cable 10ft")
print("ASIN: B082X666WZ")

USER PREFERENCE PROFILE
------------------------------------------------------------
Preferred categories:
  • Lightning Cables
  • Keyboard Skins
  • Labeling Tapes

Relevant previous preference:
  • JSAUX Apple MFi Certified Lightning Cables

CURRENT QUERY
------------------------------------------------------------
lightning cable for iPhone 12

GROUND-TRUTH PRODUCT
------------------------------------------------------------
JSAUX Apple MFi Certified Lightning Cable 10ft
ASIN: B082X666WZ


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# Project paths
PROJECT_ROOT = Path("/users/adhp511/query-conditioned-recommender")
GENERATED_DIR = PROJECT_ROOT / "data" / "generated"

# Evaluation data
EVAL_DATA_PATH = GENERATED_DIR / "proposed_model_data.parquet"

# Saved retrieval results
BLAIR_RESULTS_PATH = GENERATED_DIR / "blair_baseline_retrievals.parquet"
PROPOSED_RESULTS_PATH = GENERATED_DIR / "proposed_model_retrievals.parquet"


# Load files
eval_data = pd.read_parquet(EVAL_DATA_PATH)
blair_results = pd.read_parquet(BLAIR_RESULTS_PATH)
proposed_results = pd.read_parquet(PROPOSED_RESULTS_PATH)


print("Evaluation data shape:", eval_data.shape)
print("Query-only retrieval shape:", blair_results.shape)
print("Profile-conditioned retrieval shape:", proposed_results.shape)

print("\nEvaluation data columns:")
print(eval_data.columns.tolist())

print("\nQuery-only retrieval columns:")
print(blair_results.columns.tolist())

print("\nProfile-conditioned retrieval columns:")
print(proposed_results.columns.tolist())

Candidate examples: 37


In [2]:
# Validate that both retrieval conditions use exactly the same evaluation examples

eval_ids = set(eval_data["sample_id"])
blair_ids = set(blair_results["sample_id"])
proposed_ids = set(proposed_results["sample_id"])

print("Evaluation IDs match query-only results:",
      eval_ids == blair_ids)

print("Evaluation IDs match profile-conditioned results:",
      eval_ids == proposed_ids)

# Sort by sample_id so that rows can be compared directly
eval_check = (
    eval_data[["sample_id", "user_id", "synthetic_query", "ground_truth_asin"]]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

blair_check = (
    blair_results[["sample_id", "user_id", "synthetic_query", "ground_truth_asin"]]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

proposed_check = (
    proposed_results[["sample_id", "user_id", "synthetic_query", "ground_truth_asin"]]
    .sort_values("sample_id")
    .reset_index(drop=True)
)

print("\nQuery-only metadata matches evaluation data:",
      eval_check.equals(blair_check))

print("Profile-conditioned metadata matches evaluation data:",
      eval_check.equals(proposed_check))

print("\nDuplicate sample IDs:")
print("Evaluation:", eval_data["sample_id"].duplicated().sum())
print("Query-only:", blair_results["sample_id"].duplicated().sum())
print("Profile-conditioned:", proposed_results["sample_id"].duplicated().sum())

Evaluation IDs match query-only results: True
Evaluation IDs match profile-conditioned results: True

Query-only metadata matches evaluation data: True
Profile-conditioned metadata matches evaluation data: True

Duplicate sample IDs:
Evaluation: 0
Query-only: 0
Profile-conditioned: 0


## 11.2 Retrieval Validation

Each retrieval condition produces a ranked list of 50 products for every evaluation example. The retrieved product identifiers and their corresponding similarity scores are validated before computing ranking metrics.

The ground-truth item is considered retrieved when its product identifier appears anywhere within the Top-50 results.

In [3]:
# Validate the structure of the saved retrieval lists

def list_length(value):
    if value is None:
        return 0
    return len(value)


blair_results["num_retrieved"] = blair_results["retrieved_asins"].apply(list_length)
blair_results["num_scores"] = blair_results["retrieval_scores"].apply(list_length)

proposed_results["num_retrieved"] = proposed_results["retrieved_asins"].apply(list_length)
proposed_results["num_scores"] = proposed_results["retrieval_scores"].apply(list_length)


print("Query-only retrieval counts:")
print("  Unique numbers of retrieved ASINs:", 
      sorted(blair_results["num_retrieved"].unique()))
print("  Unique numbers of scores:",
      sorted(blair_results["num_scores"].unique()))

print("\nProfile-conditioned retrieval counts:")
print("  Unique numbers of retrieved ASINs:",
      sorted(proposed_results["num_retrieved"].unique()))
print("  Unique numbers of scores:",
      sorted(proposed_results["num_scores"].unique()))


# Check that each ASIN list and score list have matching lengths
blair_length_match = (
    blair_results["num_retrieved"] == blair_results["num_scores"]
).all()

proposed_length_match = (
    proposed_results["num_retrieved"] == proposed_results["num_scores"]
).all()

print("\nQuery-only ASIN/score lengths match:", blair_length_match)
print("Profile-conditioned ASIN/score lengths match:", proposed_length_match)


# Check whether the ground-truth item occurs in Top-50
blair_results["ground_truth_in_top50"] = blair_results.apply(
    lambda row: row["ground_truth_asin"] in row["retrieved_asins"],
    axis=1
)

proposed_results["ground_truth_in_top50"] = proposed_results.apply(
    lambda row: row["ground_truth_asin"] in row["retrieved_asins"],
    axis=1
)


print("\nGround truth retrieved in Top-50:")
print(
    "  Query-only:",
    blair_results["ground_truth_in_top50"].sum(),
    "/",
    len(blair_results)
)

print(
    "  Profile-conditioned:",
    proposed_results["ground_truth_in_top50"].sum(),
    "/",
    len(proposed_results)
)

Query-only retrieval counts:
  Unique numbers of retrieved ASINs: [np.int64(50)]
  Unique numbers of scores: [np.int64(50)]

Profile-conditioned retrieval counts:
  Unique numbers of retrieved ASINs: [np.int64(50)]
  Unique numbers of scores: [np.int64(50)]

Query-only ASIN/score lengths match: True
Profile-conditioned ASIN/score lengths match: True

Ground truth retrieved in Top-50:
  Query-only: 15 / 5559
  Profile-conditioned: 38 / 5559


## 11.3 Ranking Evaluation Metrics

The retrieval systems are evaluated using Recall, Mean Reciprocal Rank (MRR) and Normalised Discounted Cumulative Gain (NDCG) at multiple ranking cut-offs.

Each evaluation example has a single held-out ground-truth product. Recall@K measures whether this product is retrieved within the first K ranked results. MRR@K measures the reciprocal rank of the ground-truth product when it occurs within the first K results, while NDCG@K accounts for the position of the relevant product and assigns greater value to higher-ranked results.

The evaluation uses ranking cut-offs of K = 1, 5, 10, 20 and 50.

In [4]:
# Ranking metric functions for a single ground-truth item per example

K_VALUES = [1, 5, 10, 20, 50]


def get_ground_truth_rank(retrieved_asins, ground_truth_asin):
    """
    Return the 1-based rank of the ground-truth ASIN.
    Return None if it is not present in the retrieved list.
    """
    for rank, asin in enumerate(retrieved_asins, start=1):
        if asin == ground_truth_asin:
            return rank
    return None


def recall_at_k(ranks, k):
    """
    Recall@K for a single ground-truth item per example.
    """
    return np.mean([
        rank is not None and rank <= k
        for rank in ranks
    ])


def mrr_at_k(ranks, k):
    """
    Mean Reciprocal Rank@K.
    """
    reciprocal_ranks = [
        1.0 / rank
        if rank is not None and rank <= k
        else 0.0
        for rank in ranks
    ]
    return np.mean(reciprocal_ranks)


def ndcg_at_k(ranks, k):
    """
    NDCG@K for a single binary relevant item per example.
    """
    ndcg_scores = [
        1.0 / np.log2(rank + 1)
        if rank is not None and rank <= k
        else 0.0
        for rank in ranks
    ]
    return np.mean(ndcg_scores)


# Calculate the ground-truth rank for every example
blair_ranks = [
    get_ground_truth_rank(
        retrieved,
        ground_truth
    )
    for retrieved, ground_truth
    in zip(
        blair_results["retrieved_asins"],
        blair_results["ground_truth_asin"]
    )
]

proposed_ranks = [
    get_ground_truth_rank(
        retrieved,
        ground_truth
    )
    for retrieved, ground_truth
    in zip(
        proposed_results["retrieved_asins"],
        proposed_results["ground_truth_asin"]
    )
]


# Basic rank validation
print("Query-only:")
print("  Ground truths found in Top-50:", sum(rank is not None for rank in blair_ranks))
print("  Ground truths not found:", sum(rank is None for rank in blair_ranks))

print("\nProfile-conditioned:")
print("  Ground truths found in Top-50:", sum(rank is not None for rank in proposed_ranks))
print("  Ground truths not found:", sum(rank is None for rank in proposed_ranks))


# Display the first 20 ranks for a quick sanity check
print("\nFirst 20 query-only ground-truth ranks:")
print(blair_ranks[:20])

print("\nFirst 20 profile-conditioned ground-truth ranks:")
print(proposed_ranks[:20])

Query-only:
  Ground truths found in Top-50: 15
  Ground truths not found: 5544

Profile-conditioned:
  Ground truths found in Top-50: 38
  Ground truths not found: 5521

First 20 query-only ground-truth ranks:
[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]

First 20 profile-conditioned ground-truth ranks:
[None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


In [5]:
# Calculate Recall@K, MRR@K and NDCG@K for both methods

metric_rows = []

for k in K_VALUES:
    metric_rows.append({
        "K": k,
        "Recall@K (Query-only)": recall_at_k(blair_ranks, k),
        "Recall@K (Profile + Query)": recall_at_k(proposed_ranks, k),
        "MRR@K (Query-only)": mrr_at_k(blair_ranks, k),
        "MRR@K (Profile + Query)": mrr_at_k(proposed_ranks, k),
        "NDCG@K (Query-only)": ndcg_at_k(blair_ranks, k),
        "NDCG@K (Profile + Query)": ndcg_at_k(proposed_ranks, k),
    })

metrics = pd.DataFrame(metric_rows)

print(metrics.to_string(index=False))

 K  Recall@K (Query-only)  Recall@K (Profile + Query)  MRR@K (Query-only)  MRR@K (Profile + Query)  NDCG@K (Query-only)  NDCG@K (Profile + Query)
 1               0.000000                    0.000899            0.000000                 0.000899             0.000000                  0.000899
 5               0.000180                    0.002518            0.000045                 0.001526             0.000077                  0.001774
10               0.000540                    0.003778            0.000097                 0.001695             0.000198                  0.002182
20               0.001619                    0.004497            0.000176                 0.001746             0.000476                  0.002365
50               0.002698                    0.006836            0.000208                 0.001827             0.000684                  0.002840


## 11.4 Overall Retrieval Performance

The retrieval performance of the query-only and profile-conditioned approaches is compared across multiple ranking cut-offs. Absolute metric values are reported alongside the difference between the two retrieval conditions.

The comparison is performed on the same 5,559 evaluation examples and uses identical ground-truth products and ranking cut-offs.

In [6]:
# Create a compact comparison table

comparison = pd.DataFrame({
    "K": K_VALUES,
    "Recall@K - Query-only": [
        recall_at_k(blair_ranks, k) for k in K_VALUES
    ],
    "Recall@K - Profile + Query": [
        recall_at_k(proposed_ranks, k) for k in K_VALUES
    ],
    "MRR@K - Query-only": [
        mrr_at_k(blair_ranks, k) for k in K_VALUES
    ],
    "MRR@K - Profile + Query": [
        mrr_at_k(proposed_ranks, k) for k in K_VALUES
    ],
    "NDCG@K - Query-only": [
        ndcg_at_k(blair_ranks, k) for k in K_VALUES
    ],
    "NDCG@K - Profile + Query": [
        ndcg_at_k(proposed_ranks, k) for k in K_VALUES
    ],
})

# Round only for display
display(
    comparison.style
    .format({
        col: "{:.6f}"
        for col in comparison.columns
        if col != "K"
    })
)

,K,Recall@K - Query-only,Recall@K - Profile + Query,MRR@K - Query-only,MRR@K - Profile + Query,NDCG@K - Query-only,NDCG@K - Profile + Query
0,1,0.000000,0.000899,0.000000,0.000899,0.000000,0.000899
1,5,0.000180,0.002518,0.000045,0.001526,0.000077,0.001774
2,10,0.000540,0.003778,0.000097,0.001695,0.000198,0.002182
3,20,0.001619,0.004497,0.000176,0.001746,0.000476,0.002365
4,50,0.002698,0.006836,0.000208,0.001827,0.000684,0.002840


In [7]:
# Calculate absolute differences between profile-conditioned and query-only retrieval

difference_table = pd.DataFrame({
    "K": K_VALUES,
    "Recall@K Difference": [
        recall_at_k(proposed_ranks, k) - recall_at_k(blair_ranks, k)
        for k in K_VALUES
    ],
    "MRR@K Difference": [
        mrr_at_k(proposed_ranks, k) - mrr_at_k(blair_ranks, k)
        for k in K_VALUES
    ],
    "NDCG@K Difference": [
        ndcg_at_k(proposed_ranks, k) - ndcg_at_k(blair_ranks, k)
        for k in K_VALUES
    ],
})

display(
    difference_table.style.format({
        "Recall@K Difference": "{:.6f}",
        "MRR@K Difference": "{:.6f}",
        "NDCG@K Difference": "{:.6f}",
    })
)

,K,Recall@K Difference,MRR@K Difference,NDCG@K Difference
0,1,0.000899,0.000899,0.000899
1,5,0.002339,0.001481,0.001696
2,10,0.003238,0.001598,0.001984
3,20,0.002878,0.001570,0.001889
4,50,0.004137,0.001619,0.002156


## 11.5 Paired Retrieval Analysis

Because both retrieval conditions are evaluated on exactly the same examples, their outcomes can be compared on a paired basis.

For each evaluation example, the rank of the held-out ground-truth product is compared between the query-only and profile-conditioned conditions. The analysis identifies cases where profile conditioning newly retrieves the ground-truth product, cases where it removes an item retrieved by the query-only approach, and cases where both conditions produce the same retrieval status.

In [8]:
# Create a paired comparison of ground-truth ranks

paired_results = pd.DataFrame({
    "sample_id": blair_results["sample_id"],
    "user_id": blair_results["user_id"],
    "ground_truth_asin": blair_results["ground_truth_asin"],
    "query_only_rank": blair_ranks,
    "profile_query_rank": proposed_ranks,
})

# Retrieval status within Top-50
paired_results["query_only_top50"] = paired_results["query_only_rank"].notna()
paired_results["profile_query_top50"] = paired_results["profile_query_rank"].notna()


def classify_retrieval_change(row):
    query_hit = row["query_only_top50"]
    profile_hit = row["profile_query_top50"]

    if not query_hit and profile_hit:
        return "Profile + Query newly retrieves ground truth"
    elif query_hit and not profile_hit:
        return "Query-only retrieves ground truth, Profile + Query does not"
    elif query_hit and profile_hit:
        return "Both retrieve ground truth"
    else:
        return "Neither retrieves ground truth"


paired_results["retrieval_outcome"] = paired_results.apply(
    classify_retrieval_change,
    axis=1
)


outcome_counts = (
    paired_results["retrieval_outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Count")
)

outcome_counts["Percentage"] = (
    outcome_counts["Count"] / len(paired_results) * 100
)

display(
    outcome_counts.style.format({
        "Percentage": "{:.3f}%"
    })
)

,Outcome,Count,Percentage
0,Neither retrieves ground truth,5511,99.137%
1,Profile + Query newly retrieves ground truth,33,0.594%
2,"Query-only retrieves ground truth, Profile + Query does not",10,0.180%
3,Both retrieve ground truth,5,0.090%


## 11.6 Ground-Truth Rank Comparison

For examples where the ground-truth product is retrieved by at least one method, the ground-truth rank is compared between the query-only and profile-conditioned approaches.

Lower rank values indicate that the ground-truth product appears earlier in the retrieved list. Examples in which only one method retrieves the ground truth are also retained to identify cases where profile conditioning changes retrieval success.

In [9]:
# Analyse rank changes for examples where at least one method retrieves the ground truth

rank_comparison = paired_results[
    paired_results["query_only_top50"] |
    paired_results["profile_query_top50"]
].copy()

# Difference in rank where both methods retrieve the ground truth
rank_comparison["rank_difference"] = (
    rank_comparison["profile_query_rank"] -
    rank_comparison["query_only_rank"]
)

print("Examples where at least one method retrieves the ground truth:",
      len(rank_comparison))

print("\nExamples where both methods retrieve the ground truth:",
      (
          rank_comparison["query_only_top50"] &
          rank_comparison["profile_query_top50"]
      ).sum())

print("\nExamples where only query-only retrieves:",
      (
          rank_comparison["query_only_top50"] &
          ~rank_comparison["profile_query_top50"]
      ).sum())

print("\nExamples where only profile + query retrieves:",
      (
          ~rank_comparison["query_only_top50"] &
          rank_comparison["profile_query_top50"]
      ).sum())


# Rank changes among examples retrieved by both methods
both_retrieve = rank_comparison[
    rank_comparison["query_only_top50"] &
    rank_comparison["profile_query_top50"]
].copy()

if len(both_retrieve) > 0:
    improved = (
        both_retrieve["profile_query_rank"] <
        both_retrieve["query_only_rank"]
    ).sum()

    worsened = (
        both_retrieve["profile_query_rank"] >
        both_retrieve["query_only_rank"]
    ).sum()

    unchanged = (
        both_retrieve["profile_query_rank"] ==
        both_retrieve["query_only_rank"]
    ).sum()

    print("\nAmong examples retrieved by both methods:")
    print("  Profile + Query improves rank:", improved)
    print("  Profile + Query worsens rank:", worsened)
    print("  Rank unchanged:", unchanged)

    print("\nRank comparison:")
    print(
        both_retrieve[
            [
                "sample_id",
                "query_only_rank",
                "profile_query_rank",
                "rank_difference"
            ]
        ].sort_values("rank_difference").to_string(index=False)
    )

Examples where at least one method retrieves the ground truth: 48

Examples where both methods retrieve the ground truth: 5

Examples where only query-only retrieves: 10

Examples where only profile + query retrieves: 33

Among examples retrieved by both methods:
  Profile + Query improves rank: 3
  Profile + Query worsens rank: 2
  Rank unchanged: 0

Rank comparison:
 sample_id  query_only_rank  profile_query_rank  rank_difference
      4924             47.0                25.0            -22.0
      2974             21.0                 7.0            -14.0
      2983             12.0                 1.0            -11.0
      2642             16.0                26.0             10.0
      3163             19.0                42.0             23.0


## 11.7 Successful Retrieval Cases

The examples in which at least one retrieval condition identifies the held-out ground-truth product are examined qualitatively. The corresponding synthetic query, user preference profile and ground-truth metadata are retained alongside the retrieval ranks to assess how profile conditioning affects individual cases.

In [10]:
# Combine paired ranks with the evaluation metadata

successful_cases = (
    paired_results[
        paired_results["query_only_top50"] |
        paired_results["profile_query_top50"]
    ]
    .merge(
        eval_data[
            [
                "sample_id",
                "synthetic_query",
                "profile_text",
                "ground_truth_asin",
                "product_type",
                "intended_use"
            ]
        ],
        on=["sample_id", "ground_truth_asin"],
        how="left"
    )
)

# Reorder columns for easier inspection
successful_cases = successful_cases[
    [
        "sample_id",
        "synthetic_query",
        "profile_text",
        "ground_truth_asin",
        "product_type",
        "intended_use",
        "query_only_rank",
        "profile_query_rank",
        "retrieval_outcome"
    ]
].sort_values("sample_id")

print("Successful cases:", len(successful_cases))

display(
    successful_cases[
        [
            "sample_id",
            "synthetic_query",
            "product_type",
            "intended_use",
            "query_only_rank",
            "profile_query_rank",
            "retrieval_outcome"
        ]
    ].reset_index(drop=True)
)

Successful cases: 48


,sample_id,synthetic_query,product_type,intended_use,query_only_rank,profile_query_rank,retrieval_outcome
0,50,looking for more dvd-r media,dvd-r media,,NaN,10.0,Profile + Query newly retrieves ground truth
1,51,hp dvd-r discs for backup,dvd-r discs,backup,NaN,6.0,Profile + Query newly retrieves ground truth
2,221,best micro sd card reader for phone and tablet,micro sd card reader,,NaN,28.0,Profile + Query newly retrieves ground truth
3,803,best over-ear headphones for watching TV,over-ear headphones,,50.0,NaN,"Query-only retrieves ground truth, Profile + Q..."
4,999,best camera battery for nikon d5000,camera battery,,NaN,6.0,Profile + Query newly retrieves ground truth
5,1148,lightning cable for iPhone 14,cable,,NaN,1.0,Profile + Query newly retrieves ground truth
6,1383,lightning cable for iPhone 12,cable,,NaN,21.0,Profile + Query newly retrieves ground truth
7,1399,looking for a new internal hard drive to expan...,internal hard drive,laptop,NaN,24.0,Profile + Query newly retrieves ground truth
8,1551,best hard shell case for macbook air 2022,hard shell case,,NaN,16.0,Profile + Query newly retrieves ground truth
9,1552,best monitor arm for dual monitors,monitor arm,,NaN,25.0,Profile + Query newly retrieves ground truth


In [11]:
# Load product metadata only for the 48 ground-truth products in successful cases

successful_asins = successful_cases["ground_truth_asin"].dropna().unique()

catalogue_path = GENERATED_DIR / "full_product_catalogue.parquet"

ground_truth_products = pd.read_parquet(
    catalogue_path,
    filters=[
        ("parent_asin", "in", successful_asins.tolist())
    ]
)

print("Unique successful ground-truth ASINs:", len(successful_asins))
print("Ground-truth products retrieved:", len(ground_truth_products))

print("\nProduct metadata columns:")
print(ground_truth_products.columns.tolist())

display(
    ground_truth_products[
        ["parent_asin", "title", "store", "categories"]
    ].head(10)
)

Unique successful ground-truth ASINs: 47
Ground-truth products retrieved: 47

Product metadata columns:
['parent_asin', 'title', 'store', 'categories', 'product_text']


,parent_asin,title,store,categories
0,B082T7HDGT,"Amazon Basics ABS USB-A to Lightning Cable, MF...",Amazon Basics,"[Electronics, Computers & Accessories, Compute..."
1,B08TTT6F56,Zeskit Premium 3.5mm Jack Male to Female AUX A...,Zeskit,"[Electronics, Home Audio, Home Audio Accessori..."
2,B0BRXKZHGM,OREI HDMI Multi-Viewer 4x1 Seamless HDMI Switc...,OREI,"[Electronics, Home Audio, Home Audio Accessori..."
3,B009WZRCO6,"Verso ""Profile"" Standing Cover for Kindle Fire...",Lightwedge,"[Electronics, Computers & Accessories, Tablet ..."
4,B07S8F43Y6,"D-Line Cord Cover 2-Pack, Cord Hiders, Cable C...",D-Line,"[Electronics, Accessories & Supplies, Cord Man..."
5,B09DG877JS,"Type C Cable 3A Fast Charging, Conmdex [2-Pack...",CONMDEX,"[Electronics, Computers & Accessories, Compute..."
6,B0BZT5GQGJ,"Amazon Basics HDMI Cable, 48Gbps High-Speed, 8...",Amazon Basics,"[Electronics, Television & Video, Accessories,..."
7,B0BZVF3W2L,JETech Screen Protector for iPad mini 6 (8.3-I...,JETech,"[Electronics, Computers & Accessories, Tablet ..."
8,B09G35YJGN,Wearlizer Compatible with Fitbit Charge 3 / Ch...,Wearlizer,"[Electronics, Wearable Technology, Arm & Wrist..."
9,B0BHXQLLXB,TALK WORKS Flat Lightning Charging Cable Compa...,TALK WORKS,"[Electronics, Computers & Accessories, Compute..."


In [12]:
# Add ground-truth product titles to the successful-case analysis

successful_cases_with_products = successful_cases.merge(
    ground_truth_products[
        ["parent_asin", "title", "store", "categories"]
    ],
    left_on="ground_truth_asin",
    right_on="parent_asin",
    how="left"
)

successful_cases_with_products = successful_cases_with_products.drop(
    columns=["parent_asin"]
)

successful_cases_with_products = successful_cases_with_products.rename(
    columns={
        "title": "ground_truth_title",
        "store": "ground_truth_store",
        "categories": "ground_truth_categories"
    }
)

# Display the most informative fields
display(
    successful_cases_with_products[
        [
            "sample_id",
            "synthetic_query",
            "product_type",
            "intended_use",
            "ground_truth_title",
            "query_only_rank",
            "profile_query_rank",
            "retrieval_outcome"
        ]
    ].reset_index(drop=True)
)

,sample_id,synthetic_query,product_type,intended_use,ground_truth_title,query_only_rank,profile_query_rank,retrieval_outcome
0,50,looking for more dvd-r media,dvd-r media,,PHILIPS Value Pack Dm4s6b00f/17 16X DVD-R 100/...,NaN,10.0,Profile + Query newly retrieves ground truth
1,51,hp dvd-r discs for backup,dvd-r discs,backup,Smart Buy 100 Pack DVD-R 4.7gb 16x Logo Blank ...,NaN,6.0,Profile + Query newly retrieves ground truth
2,221,best micro sd card reader for phone and tablet,micro sd card reader,,UNITEK Dual Option USB-C Type C (USB 3.1 Gen 1...,NaN,28.0,Profile + Query newly retrieves ground truth
3,803,best over-ear headphones for watching TV,over-ear headphones,,NY Over Ear Best Stereo Extra Bass Portable He...,50.0,NaN,"Query-only retrieves ground truth, Profile + Q..."
4,999,best camera battery for nikon d5000,camera battery,,Echo Show 8 (2nd Gen) Adjustable Stand | Glaci...,NaN,6.0,Profile + Query newly retrieves ground truth
5,1148,lightning cable for iPhone 14,cable,,TALK WORKS Flat Lightning Charging Cable Compa...,NaN,1.0,Profile + Query newly retrieves ground truth
6,1383,lightning cable for iPhone 12,cable,,JSAUX 【Apple MFi Certified】 Lightning Cable 10...,NaN,21.0,Profile + Query newly retrieves ground truth
7,1399,looking for a new internal hard drive to expan...,internal hard drive,laptop,TRENDnet 4-Port PS2 KVM Switch and Cable Kit w...,NaN,24.0,Profile + Query newly retrieves ground truth
8,1551,best hard shell case for macbook air 2022,hard shell case,,ASURION 3 Year Office Equipment Protection Pla...,NaN,16.0,Profile + Query newly retrieves ground truth
9,1552,best monitor arm for dual monitors,monitor arm,,ASURION 3 Year Office Equipment Protection Pla...,NaN,25.0,Profile + Query newly retrieves ground truth


## 11.8 Query–Ground-Truth Consistency

The synthetic evaluation procedure assumes that the held-out product associated with an interaction represents the relevant product for the generated query. However, synthetic query generation may produce queries that do not accurately describe the held-out product.

To assess the validity of the evaluation ground truth, the generated queries are compared with the corresponding ground-truth product metadata. This analysis distinguishes retrieval performance from potential inconsistencies introduced during synthetic query construction.

In [13]:
# Prepare query–ground-truth consistency table

consistency_table = successful_cases_with_products[
    [
        "sample_id",
        "synthetic_query",
        "product_type",
        "intended_use",
        "ground_truth_title",
        "ground_truth_store",
        "ground_truth_categories",
        "query_only_rank",
        "profile_query_rank",
        "retrieval_outcome"
    ]
].copy()

display(consistency_table.reset_index(drop=True))

,sample_id,synthetic_query,product_type,intended_use,ground_truth_title,ground_truth_store,ground_truth_categories,query_only_rank,profile_query_rank,retrieval_outcome
0,50,looking for more dvd-r media,dvd-r media,,PHILIPS Value Pack Dm4s6b00f/17 16X DVD-R 100/...,Philips,"[Electronics, Computers & Accessories, Compute...",NaN,10.0,Profile + Query newly retrieves ground truth
1,51,hp dvd-r discs for backup,dvd-r discs,backup,Smart Buy 100 Pack DVD-R 4.7gb 16x Logo Blank ...,Smartbuy,"[Electronics, Computers & Accessories, Compute...",NaN,6.0,Profile + Query newly retrieves ground truth
2,221,best micro sd card reader for phone and tablet,micro sd card reader,,UNITEK Dual Option USB-C Type C (USB 3.1 Gen 1...,Unitek,"[Electronics, Computers & Accessories, Compute...",NaN,28.0,Profile + Query newly retrieves ground truth
3,803,best over-ear headphones for watching TV,over-ear headphones,,NY Over Ear Best Stereo Extra Bass Portable He...,Sony,"[Electronics, Headphones, Earbuds & Accessorie...",50.0,NaN,"Query-only retrieves ground truth, Profile + Q..."
4,999,best camera battery for nikon d5000,camera battery,,Echo Show 8 (2nd Gen) Adjustable Stand | Glaci...,Amazon,[],NaN,6.0,Profile + Query newly retrieves ground truth
5,1148,lightning cable for iPhone 14,cable,,TALK WORKS Flat Lightning Charging Cable Compa...,TALK WORKS,"[Electronics, Computers & Accessories, Compute...",NaN,1.0,Profile + Query newly retrieves ground truth
6,1383,lightning cable for iPhone 12,cable,,JSAUX 【Apple MFi Certified】 Lightning Cable 10...,JSAUX,"[Electronics, Computers & Accessories, Compute...",NaN,21.0,Profile + Query newly retrieves ground truth
7,1399,looking for a new internal hard drive to expan...,internal hard drive,laptop,TRENDnet 4-Port PS2 KVM Switch and Cable Kit w...,TRENDnet,"[Electronics, Computers & Accessories, Compute...",NaN,24.0,Profile + Query newly retrieves ground truth
8,1551,best hard shell case for macbook air 2022,hard shell case,,ASURION 3 Year Office Equipment Protection Pla...,ASURION,"[Electronics, Electronics Warranties]",NaN,16.0,Profile + Query newly retrieves ground truth
9,1552,best monitor arm for dual monitors,monitor arm,,ASURION 3 Year Office Equipment Protection Pla...,ASURION,"[Electronics, Electronics Warranties]",NaN,25.0,Profile + Query newly retrieves ground truth


In [14]:
# Summarise ground-truth rank distributions

def rank_summary(ranks, method_name):
    valid_ranks = [rank for rank in ranks if rank is not None]

    return {
        "Method": method_name,
        "Hits@50": len(valid_ranks),
        "Hits@1": sum(rank <= 1 for rank in valid_ranks),
        "Hits@5": sum(rank <= 5 for rank in valid_ranks),
        "Hits@10": sum(rank <= 10 for rank in valid_ranks),
        "Hits@20": sum(rank <= 20 for rank in valid_ranks),
        "Mean rank (hits)": np.mean(valid_ranks) if valid_ranks else np.nan,
        "Median rank (hits)": np.median(valid_ranks) if valid_ranks else np.nan,
    }


rank_summary_table = pd.DataFrame([
    rank_summary(blair_ranks, "Query-only"),
    rank_summary(proposed_ranks, "Profile + Query")
])

display(
    rank_summary_table.style.format({
        "Mean rank (hits)": "{:.2f}",
        "Median rank (hits)": "{:.1f}",
    })
)

,Method,Hits@50,Hits@1,Hits@5,Hits@10,Hits@20,Mean rank (hits),Median rank (hits)
0,Query-only,15,0,1,3,9,21.93,16.0
1,Profile + Query,38,5,14,21,25,14.24,9.0


## 11.9 Saving Evaluation Results

The aggregate metrics and paired retrieval analysis are saved as intermediate evaluation outputs. These files provide reproducible records of the quantitative comparison and per-example ground-truth ranks.

In [15]:
# Save quantitative evaluation results

METRICS_PATH = GENERATED_DIR / "retrieval_metrics.csv"
DIFFERENCE_PATH = GENERATED_DIR / "retrieval_metric_differences.csv"
PAIRED_PATH = GENERATED_DIR / "paired_retrieval_analysis.parquet"
RANK_SUMMARY_PATH = GENERATED_DIR / "retrieval_rank_summary.csv"


# Save aggregate metrics
metrics.to_csv(METRICS_PATH, index=False)

# Save metric differences
difference_table.to_csv(DIFFERENCE_PATH, index=False)

# Save paired per-example analysis
paired_results.to_parquet(PAIRED_PATH, index=False)

# Save rank summary
rank_summary_table.to_csv(RANK_SUMMARY_PATH, index=False)


print("Saved evaluation outputs:")
for path in [
    METRICS_PATH,
    DIFFERENCE_PATH,
    PAIRED_PATH,
    RANK_SUMMARY_PATH
]:
    print(f"  {path.name}: {path.stat().st_size / (1024**2):.3f} MB")

Saved evaluation outputs:
  retrieval_metrics.csv: 0.001 MB
  retrieval_metric_differences.csv: 0.000 MB
  paired_retrieval_analysis.parquet: 0.103 MB
  retrieval_rank_summary.csv: 0.000 MB


## 11.10 Qualitative Retrieval Analysis

Quantitative metrics provide an overall comparison, but they do not show how incorporating user preference information changes individual retrieval results. A qualitative analysis is therefore conducted using paired examples from the same evaluation set.

Examples are selected from three outcome groups: cases where profile conditioning newly retrieves the ground-truth product, cases where profile conditioning causes a previously retrieved ground-truth product to be lost, and cases where both methods retrieve the ground truth but at different ranks.

The analysis considers the relationship between the synthetic query, user preference profile, ground-truth product and retrieved ranking. Examples are selected to illustrate both beneficial and adverse effects of profile conditioning rather than only favourable outcomes.

In [16]:
# Select candidate examples for balanced qualitative analysis

# 1. Profile + Query newly retrieves the ground truth
new_profile_hits = successful_cases_with_products[
    successful_cases_with_products["retrieval_outcome"]
    == "Profile + Query newly retrieves ground truth"
].copy()

# 2. Query-only retrieves but Profile + Query does not
lost_query_hits = successful_cases_with_products[
    successful_cases_with_products["retrieval_outcome"]
    == "Query-only retrieves ground truth, Profile + Query does not"
].copy()

# 3. Both retrieve the ground truth
both_hits = successful_cases_with_products[
    successful_cases_with_products["retrieval_outcome"]
    == "Both retrieve ground truth"
].copy()


print("Profile + Query newly retrieves:", len(new_profile_hits))
print("Query-only retrieves but Profile + Query does not:", len(lost_query_hits))
print("Both retrieve:", len(both_hits))


print("\n--- Profile + Query newly retrieves: highest-ranked cases ---")
display(
    new_profile_hits[
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_title",
            "profile_query_rank"
        ]
    ]
    .sort_values("profile_query_rank")
    .head(10)
    .reset_index(drop=True)
)


print("\n--- Query-only retrieves but Profile + Query does not ---")
display(
    lost_query_hits[
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_title",
            "query_only_rank"
        ]
    ]
    .sort_values("query_only_rank")
    .head(10)
    .reset_index(drop=True)
)


print("\n--- Both retrieve: rank comparison ---")
display(
    both_hits[
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_title",
            "query_only_rank",
            "profile_query_rank"
        ]
    ]
    .sort_values("query_only_rank")
    .reset_index(drop=True)
)

Profile + Query newly retrieves: 33
Query-only retrieves but Profile + Query does not: 10
Both retrieve: 5

--- Profile + Query newly retrieves: highest-ranked cases ---


,sample_id,synthetic_query,ground_truth_title,profile_query_rank
0,1148,lightning cable for iPhone 14,TALK WORKS Flat Lightning Charging Cable Compa...,1.0
1,2293,looking for a new on-ear headphone with good s...,Zeskit Premium 3.5mm Jack Male to Female AUX A...,1.0
2,4906,iPhone 13 case with good reviews,Fintie Hybrid Slim Case for iPad Pro 12.9-inch...,1.0
3,2947,best 10mp digital camera under $200,Panasonic Lumix DMC-ZS3 10.1 MP Digital Camera...,1.0
4,2681,HDMI extender for long distance connection,OREI HDMI Multi-Viewer 4x1 Seamless HDMI Switc...,2.0
5,3580,best screen protector for macbook air,Matone Outlet Wall Mount Holder for 3rd Genera...,2.0
6,2147,Looking for another external hard drive like t...,"D-Line Cord Cover 2-Pack, Cord Hiders, Cable C...",2.0
7,2641,photography backdrop for portrait shooting,"econious Photography Backdrop, 8x10ft Gradient...",2.0
8,3671,looking for a high-quality power strip for my ...,"Mind Reader Monitor Stand, Monitor Riser for C...",2.0
9,2004,Bose soundbar with hdmi cable,"Amazon Basics ABS USB-A to Lightning Cable, MF...",3.0



--- Query-only retrieves but Profile + Query does not ---


,sample_id,synthetic_query,ground_truth_title,query_only_rank
0,4566,looking for a warranty plan for my new fire tv...,2-Year Protection Plan for Fire TV Stick 4K,4.0
1,2056,looking for another high-speed hdmi cable like...,"Amazon Basics HDMI Cable, 48Gbps High-Speed, 8...",6.0
2,2986,looking for another usb wifi adapter like tp-l...,TP-Link Archer T2U 11AC USB WiFi Adapter - Dua...,8.0
3,4111,best screen protector for ipadmini 6,JETech Screen Protector for iPad mini 6 (8.3-I...,11.0
4,4563,fire tv stick alexa voice remote,Fire TV Stick (3rd Gen) with Alexa Voice Remot...,12.0
5,3335,best usb c cable for s22,"Type C Cable 3A Fast Charging, Conmdex [2-Pack...",15.0
6,3242,fitbit charge 4 replacement band,Wearlizer Compatible with Fitbit Charge 3 / Ch...,32.0
7,2982,usb flash drive for hp sprout,Sprout by HP 23-Inch Touchscreen Desktop - Int...,34.0
8,2516,replacement batteries for arlo pro cameras,Arlo Rechargeable Battery - Arlo Certified Acc...,42.0
9,803,best over-ear headphones for watching TV,NY Over Ear Best Stereo Extra Bass Portable He...,50.0



--- Both retrieve: rank comparison ---


,sample_id,synthetic_query,ground_truth_title,query_only_rank,profile_query_rank
0,2983,hp sprout 23 inch touchscreen desktop sleeve,HP Sprout with HP Illuminator 23'' FHD touch s...,12.0,1.0
1,2642,econious photography backdrop 6x8ft abstract art,"econious Photography Backdrop, 5x7ft Abstract ...",16.0,26.0
2,3163,archival storage pages for 4x6 prints,"Archival Photo Pages Holds Two 8 x 10"" Prints,...",19.0,42.0
3,2974,looking for another tp-link usb wifi adapter,TP-Link Archer T2U 11AC USB WiFi Adapter - Dua...,21.0,7.0
4,4924,looking for a replacement remote control simil...,Logitech Harmony 700 Rechargeable Remote with ...,47.0,25.0


## 11.11 Candidate Selection for Qualitative Analysis

To avoid selecting qualitative examples solely on the basis of retrieval rank, candidate cases are identified using simple lexical overlap between the synthetic query and the available ground-truth product metadata.

This procedure is used only to identify examples for manual inspection and is not used to calculate retrieval metrics or modify the evaluation dataset. Final qualitative examples are assessed manually based on the consistency between the query, product and user context.

In [17]:
# Identify qualitative-analysis candidates using simple lexical overlap.
# This is ONLY a selection aid and is not an evaluation metric.

import re


def normalise_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).split()


def overlap_score(query, title, categories):
    query_words = set(normalise_text(query))
    product_words = set(normalise_text(title))

    if isinstance(categories, list):
        category_text = " ".join(str(x) for x in categories)
    else:
        category_text = str(categories)

    product_words.update(normalise_text(category_text))

    # Ignore very common generic words
    stop_words = {
        "the", "a", "an", "for", "with", "and", "or", "to",
        "my", "looking", "best", "good", "new", "another",
        "like", "under", "on", "in", "of"
    }

    query_words = query_words - stop_words
    product_words = product_words - stop_words

    if not query_words:
        return 0.0

    return len(query_words & product_words) / len(query_words)


successful_cases_with_products["lexical_overlap"] = successful_cases_with_products.apply(
    lambda row: overlap_score(
        row["synthetic_query"],
        row["ground_truth_title"],
        row["ground_truth_categories"]
    ),
    axis=1
)


# Show highest-overlap examples from each outcome group

for outcome_name, group in [
    (
        "Profile + Query newly retrieves",
        new_profile_hits
    ),
    (
        "Query-only retrieves but Profile + Query does not",
        lost_query_hits
    ),
    (
        "Both retrieve",
        both_hits
    )
]:
    candidates = successful_cases_with_products[
        successful_cases_with_products["retrieval_outcome"] == outcome_name
    ].sort_values(
        ["lexical_overlap", "profile_query_rank"],
        ascending=[False, True]
    )

    print(f"\n--- {outcome_name} ---")

    display(
        candidates[
            [
                "sample_id",
                "synthetic_query",
                "ground_truth_title",
                "lexical_overlap",
                "query_only_rank",
                "profile_query_rank"
            ]
        ]
        .head(10)
        .reset_index(drop=True)
    )


--- Profile + Query newly retrieves ---


,sample_id,synthetic_query,ground_truth_title,lexical_overlap,query_only_rank,profile_query_rank



--- Query-only retrieves but Profile + Query does not ---


,sample_id,synthetic_query,ground_truth_title,lexical_overlap,query_only_rank,profile_query_rank



--- Both retrieve ---


,sample_id,synthetic_query,ground_truth_title,lexical_overlap,query_only_rank,profile_query_rank


In [18]:
# Check the outcome labels currently present in the qualitative-analysis table

print("Number of rows:", len(successful_cases_with_products))

print("\nRetrieval outcome values:")
print(
    successful_cases_with_products["retrieval_outcome"]
    .value_counts(dropna=False)
)

print("\nLexical overlap summary:")
print(
    successful_cases_with_products["lexical_overlap"].describe()
)

Number of rows: 48

Retrieval outcome values:
retrieval_outcome
Profile + Query newly retrieves ground truth                   33
Query-only retrieves ground truth, Profile + Query does not    10
Both retrieve ground truth                                      5
Name: count, dtype: int64

Lexical overlap summary:
count    48.000000
mean      0.408383
std       0.385387
min       0.000000
25%       0.000000
50%       0.366667
75%       0.750000
max       1.000000
Name: lexical_overlap, dtype: float64


In [19]:
# Build clean candidate tables directly from the current analysis dataframe

profile_help_candidates = (
    successful_cases_with_products[
        successful_cases_with_products["retrieval_outcome"]
        == "Profile + Query newly retrieves ground truth"
    ]
    .sort_values(
        ["lexical_overlap", "profile_query_rank"],
        ascending=[False, True]
    )
    .head(10)
)

profile_hurt_candidates = (
    successful_cases_with_products[
        successful_cases_with_products["retrieval_outcome"]
        == "Query-only retrieves ground truth, Profile + Query does not"
    ]
    .sort_values(
        ["lexical_overlap", "query_only_rank"],
        ascending=[False, True]
    )
    .head(10)
)

both_candidates = (
    successful_cases_with_products[
        successful_cases_with_products["retrieval_outcome"]
        == "Both retrieve ground truth"
    ]
    .sort_values("lexical_overlap", ascending=False)
)


print("Profile + Query newly retrieves — candidates:")
display(
    profile_help_candidates[
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_title",
            "lexical_overlap",
            "query_only_rank",
            "profile_query_rank"
        ]
    ].reset_index(drop=True)
)


print("\nQuery-only retrieves but Profile + Query does not — candidates:")
display(
    profile_hurt_candidates[
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_title",
            "lexical_overlap",
            "query_only_rank",
            "profile_query_rank"
        ]
    ].reset_index(drop=True)
)


print("\nBoth retrieve — candidates:")
display(
    both_candidates[
        [
            "sample_id",
            "synthetic_query",
            "ground_truth_title",
            "lexical_overlap",
            "query_only_rank",
            "profile_query_rank"
        ]
    ].reset_index(drop=True)
)

Profile + Query newly retrieves — candidates:


,sample_id,synthetic_query,ground_truth_title,lexical_overlap,query_only_rank,profile_query_rank
0,1148,lightning cable for iPhone 14,TALK WORKS Flat Lightning Charging Cable Compa...,1.000000,NaN,1.0
1,2560,replacement camera battery for kodak easyshare,Kastar Battery x2 and Slim LCD Charger Replace...,1.000000,NaN,36.0
2,2641,photography backdrop for portrait shooting,"econious Photography Backdrop, 8x10ft Gradient...",0.750000,NaN,2.0
3,50,looking for more dvd-r media,PHILIPS Value Pack Dm4s6b00f/17 16X DVD-R 100/...,0.750000,NaN,10.0
4,1383,lightning cable for iPhone 12,JSAUX 【Apple MFi Certified】 Lightning Cable 10...,0.750000,NaN,21.0
5,221,best micro sd card reader for phone and tablet,UNITEK Dual Option USB-C Type C (USB 3.1 Gen 1...,0.666667,NaN,28.0
6,2947,best 10mp digital camera under $200,Panasonic Lumix DMC-ZS3 10.1 MP Digital Camera...,0.500000,NaN,1.0
7,4614,Best TV antenna for outdoor use and long range,"TV Antenna, Vansky Outdoor Amplified Digital H...",0.500000,NaN,44.0
8,51,hp dvd-r discs for backup,Smart Buy 100 Pack DVD-R 4.7gb 16x Logo Blank ...,0.400000,NaN,6.0
9,4148,Z-Wave motion detector with pet immunity,GoControl WNK01-21KIT Essential Z-Wave Home Se...,0.333333,NaN,15.0



Query-only retrieves but Profile + Query does not — candidates:


,sample_id,synthetic_query,ground_truth_title,lexical_overlap,query_only_rank,profile_query_rank
0,2986,looking for another usb wifi adapter like tp-l...,TP-Link Archer T2U 11AC USB WiFi Adapter - Dua...,1.000000,8.0,NaN
1,4563,fire tv stick alexa voice remote,Fire TV Stick (3rd Gen) with Alexa Voice Remot...,1.000000,12.0,NaN
2,3335,best usb c cable for s22,"Type C Cable 3A Fast Charging, Conmdex [2-Pack...",1.000000,15.0,NaN
3,3242,fitbit charge 4 replacement band,Wearlizer Compatible with Fitbit Charge 3 / Ch...,1.000000,32.0,NaN
4,2056,looking for another high-speed hdmi cable like...,"Amazon Basics HDMI Cable, 48Gbps High-Speed, 8...",0.857143,6.0,NaN
5,4566,looking for a warranty plan for my new fire tv...,2-Year Protection Plan for Fire TV Stick 4K,0.800000,4.0,NaN
6,2516,replacement batteries for arlo pro cameras,Arlo Rechargeable Battery - Arlo Certified Acc...,0.800000,42.0,NaN
7,4111,best screen protector for ipadmini 6,JETech Screen Protector for iPad mini 6 (8.3-I...,0.750000,11.0,NaN
8,2982,usb flash drive for hp sprout,Sprout by HP 23-Inch Touchscreen Desktop - Int...,0.600000,34.0,NaN
9,803,best over-ear headphones for watching TV,NY Over Ear Best Stereo Extra Bass Portable He...,0.600000,50.0,NaN



Both retrieve — candidates:


,sample_id,synthetic_query,ground_truth_title,lexical_overlap,query_only_rank,profile_query_rank
0,2974,looking for another tp-link usb wifi adapter,TP-Link Archer T2U 11AC USB WiFi Adapter - Dua...,1.000000,21.0,7.0
1,3163,archival storage pages for 4x6 prints,"Archival Photo Pages Holds Two 8 x 10"" Prints,...",0.800000,19.0,42.0
2,2642,econious photography backdrop 6x8ft abstract art,"econious Photography Backdrop, 5x7ft Abstract ...",0.666667,16.0,26.0
3,4924,looking for a replacement remote control simil...,Logitech Harmony 700 Rechargeable Remote with ...,0.500000,47.0,25.0
4,2983,hp sprout 23 inch touchscreen desktop sleeve,HP Sprout with HP Illuminator 23'' FHD touch s...,0.428571,12.0,1.0


## 11.12 Ablation Results

Additional controlled experiments were conducted to examine the effect of the semantic encoder's input length and the ordering of the profile and query.

The primary proposed configuration combines the user preference profile with the synthetic query using a maximum sequence length of 256 tokens. Two ablations were evaluated using the same E5-base-v2 encoder, full product catalogue, FAISS index and evaluation set.

The first increases the maximum sequence length to 512 tokens while retaining the profile-first ordering. The second uses 512 tokens but places the current query before the historical profile.

These experiments are treated as ablations rather than replacements for the original experimental conditions. Their purpose is to determine whether input length and ordering materially affect retrieval performance.

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/users/adhp511/query-conditioned-recommender"
)

DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Data directory exists:", DATA_DIR.exists())

Project root: /users/adhp511/query-conditioned-recommender
Data directory: /users/adhp511/query-conditioned-recommender/data
Data directory exists: True


In [23]:
# Load all retrieval results

BLAIR_PATH = (
    DATA_DIR
    / "generated"
    / "blair_baseline_retrievals.parquet"
)

PROPOSED_PATH = (
    DATA_DIR
    / "generated"
    / "proposed_model_retrievals.parquet"
)

ABLATION_PROFILE_QUERY_512_PATH = (
    DATA_DIR
    / "generated"
    / "proposed_ablation_profile_query_512.parquet"
)

ABLATION_QUERY_PROFILE_512_PATH = (
    DATA_DIR
    / "generated"
    / "proposed_ablation_query_profile_512.parquet"
)

query_only_retrievals = pd.read_parquet(BLAIR_PATH)
proposed_retrievals = pd.read_parquet(PROPOSED_PATH)
profile_query_512_retrievals = pd.read_parquet(
    ABLATION_PROFILE_QUERY_512_PATH
)
query_profile_512_retrievals = pd.read_parquet(
    ABLATION_QUERY_PROFILE_512_PATH
)

print("Query-only:", query_only_retrievals.shape)
print("Profile + Query (256):", proposed_retrievals.shape)
print("Profile + Query (512):", profile_query_512_retrievals.shape)
print("Query + Profile (512):", query_profile_512_retrievals.shape)

Query-only: (5559, 6)
Profile + Query (256): (5559, 7)
Profile + Query (512): (5559, 7)
Query + Profile (512): (5559, 7)


In [24]:
# Validate that all four retrieval sets use the same evaluation examples

retrieval_sets = {
    "Query-only": query_only_retrievals,
    "Profile + Query (256)": proposed_retrievals,
    "Profile + Query (512)": profile_query_512_retrievals,
    "Query + Profile (512)": query_profile_512_retrievals,
}

reference_ids = set(query_only_retrievals["sample_id"])

for name, df in retrieval_sets.items():
    ids = set(df["sample_id"])

    print(
        f"{name}: "
        f"{len(ids)} unique IDs | "
        f"Matches query-only: {ids == reference_ids}"
    )

    print(
        f"  Duplicate sample IDs: "
        f"{df['sample_id'].duplicated().sum()}"
    )

    print(
        f"  Missing ground truth: "
        f"{df['ground_truth_asin'].isna().sum()}"
    )

Query-only: 5559 unique IDs | Matches query-only: True
  Duplicate sample IDs: 0
  Missing ground truth: 0
Profile + Query (256): 5559 unique IDs | Matches query-only: True
  Duplicate sample IDs: 0
  Missing ground truth: 0
Profile + Query (512): 5559 unique IDs | Matches query-only: True
  Duplicate sample IDs: 0
  Missing ground truth: 0
Query + Profile (512): 5559 unique IDs | Matches query-only: True
  Duplicate sample IDs: 0
  Missing ground truth: 0


In [25]:
# Final metric comparison across all retrieval conditions

K_VALUES = [1, 5, 10, 20, 50]


def get_ground_truth_rank(retrieved_asins, ground_truth_asin):
    for rank, asin in enumerate(retrieved_asins, start=1):
        if asin == ground_truth_asin:
            return rank
    return None


def recall_at_k(retrieved_asins, ground_truth_asin, k):
    rank = get_ground_truth_rank(
        retrieved_asins,
        ground_truth_asin
    )
    return int(rank is not None and rank <= k)


def mrr_at_k(retrieved_asins, ground_truth_asin, k):
    rank = get_ground_truth_rank(
        retrieved_asins,
        ground_truth_asin
    )

    if rank is not None and rank <= k:
        return 1.0 / rank

    return 0.0


def ndcg_at_k(retrieved_asins, ground_truth_asin, k):
    rank = get_ground_truth_rank(
        retrieved_asins,
        ground_truth_asin
    )

    if rank is not None and rank <= k:
        return 1.0 / np.log2(rank + 1)

    return 0.0


conditions = {
    "Query-only": query_only_retrievals,
    "Profile + Query (256)": proposed_retrievals,
    "Profile + Query (512)": profile_query_512_retrievals,
    "Query + Profile (512)": query_profile_512_retrievals,
}


results = []

for condition_name, retrievals in conditions.items():

    for k in K_VALUES:

        recall_values = []
        mrr_values = []
        ndcg_values = []

        for _, row in retrievals.iterrows():

            retrieved = row["retrieved_asins"]
            ground_truth = row["ground_truth_asin"]

            recall_values.append(
                recall_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

            mrr_values.append(
                mrr_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

            ndcg_values.append(
                ndcg_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

        results.append({
            "Condition": condition_name,
            "K": k,
            "Recall@K": np.mean(recall_values),
            "MRR@K": np.mean(mrr_values),
            "NDCG@K": np.mean(ndcg_values)
        })


final_metrics = pd.DataFrame(results)

print(final_metrics.to_string(index=False))

            Condition  K  Recall@K    MRR@K   NDCG@K
           Query-only  1  0.000000 0.000000 0.000000
           Query-only  5  0.000180 0.000045 0.000077
           Query-only 10  0.000540 0.000097 0.000198
           Query-only 20  0.001619 0.000176 0.000476
           Query-only 50  0.002698 0.000208 0.000684
Profile + Query (256)  1  0.000899 0.000899 0.000899
Profile + Query (256)  5  0.002518 0.001526 0.001774
Profile + Query (256) 10  0.003778 0.001695 0.002182
Profile + Query (256) 20  0.004497 0.001746 0.002365
Profile + Query (256) 50  0.006836 0.001827 0.002840
Profile + Query (512)  1  0.000540 0.000540 0.000540
Profile + Query (512)  5  0.002518 0.001280 0.001589
Profile + Query (512) 10  0.003958 0.001475 0.002057
Profile + Query (512) 20  0.005037 0.001547 0.002327
Profile + Query (512) 50  0.007735 0.001634 0.002865
Query + Profile (512)  1  0.000720 0.000720 0.000720
Query + Profile (512)  5  0.001799 0.001022 0.001209
Query + Profile (512) 10  0.002878 0.001164 0.

## 11.X Retrieval Hit Counts

Because the evaluation contains 5,559 examples, the metric values are small in absolute terms. Hit counts are therefore also reported to provide an interpretable view of how many evaluation examples contain the held-out ground-truth product within each retrieval cut-off.

In [26]:
# Calculate ground-truth hit counts at each retrieval cut-off

hit_count_results = []

for condition_name, retrievals in conditions.items():

    for k in K_VALUES:

        hits = 0

        for _, row in retrievals.iterrows():

            rank = get_ground_truth_rank(
                row["retrieved_asins"],
                row["ground_truth_asin"]
            )

            if rank is not None and rank <= k:
                hits += 1

        hit_count_results.append({
            "Condition": condition_name,
            "K": k,
            "Hits": hits,
            "Total examples": len(retrievals),
            "Hit rate (%)": 100 * hits / len(retrievals)
        })

hit_counts = pd.DataFrame(hit_count_results)

print(hit_counts.to_string(index=False))

            Condition  K  Hits  Total examples  Hit rate (%)
           Query-only  1     0            5559      0.000000
           Query-only  5     1            5559      0.017989
           Query-only 10     3            5559      0.053967
           Query-only 20     9            5559      0.161900
           Query-only 50    15            5559      0.269833
Profile + Query (256)  1     5            5559      0.089944
Profile + Query (256)  5    14            5559      0.251844
Profile + Query (256) 10    21            5559      0.377766
Profile + Query (256) 20    25            5559      0.449721
Profile + Query (256) 50    38            5559      0.683576
Profile + Query (512)  1     3            5559      0.053967
Profile + Query (512)  5    14            5559      0.251844
Profile + Query (512) 10    22            5559      0.395755
Profile + Query (512) 20    28            5559      0.503688
Profile + Query (512) 50    43            5559      0.773520
Query + Profile (512)  1

## 11.X Paired Retrieval Outcomes

Aggregate metrics indicate the overall retrieval performance of each method, but they do not show how the methods differ on individual evaluation examples.

A paired analysis is therefore performed using the same 5,559 examples. For each comparison, an example is classified according to whether the ground-truth product is retrieved within the Top-50 by neither method, both methods, only the first method, or only the second method.

This analysis provides a direct examination of whether profile-conditioned retrieval changes the set of examples for which the held-out ground-truth product is retrieved.

In [27]:
# Paired Top-50 retrieval outcomes

def top_k_hit(row, k=50):
    rank = get_ground_truth_rank(
        row["retrieved_asins"],
        row["ground_truth_asin"]
    )
    return rank is not None and rank <= k


def paired_outcomes(df_a, df_b, name_a, name_b, k=50):
    merged = df_a[
        ["sample_id", "ground_truth_asin", "retrieved_asins"]
    ].merge(
        df_b[
            ["sample_id", "retrieved_asins"]
        ],
        on="sample_id",
        suffixes=("_a", "_b")
    )

    hit_a = merged.apply(
        lambda row: get_ground_truth_rank(
            row["retrieved_asins_a"],
            row["ground_truth_asin"]
        ) is not None
        and get_ground_truth_rank(
            row["retrieved_asins_a"],
            row["ground_truth_asin"]
        ) <= k,
        axis=1
    )

    hit_b = merged.apply(
        lambda row: get_ground_truth_rank(
            row["retrieved_asins_b"],
            row["ground_truth_asin"]
        ) is not None
        and get_ground_truth_rank(
            row["retrieved_asins_b"],
            row["ground_truth_asin"]
        ) <= k,
        axis=1
    )

    outcomes = pd.Series(
        np.select(
            [
                (~hit_a) & (~hit_b),
                (~hit_a) & hit_b,
                hit_a & (~hit_b),
                hit_a & hit_b
            ],
            [
                "Neither retrieves",
                f"Only {name_b} retrieves",
                f"Only {name_a} retrieves",
                "Both retrieve"
            ],
            default="Unknown"
        )
    )

    summary = (
        outcomes.value_counts()
        .rename_axis("Outcome")
        .reset_index(name="Count")
    )

    summary["Percentage"] = (
        100 * summary["Count"] / len(merged)
    )

    return summary


# Query-only vs original proposed model
paired_query_vs_256 = paired_outcomes(
    query_only_retrievals,
    proposed_retrievals,
    "Query-only",
    "Profile + Query (256)"
)

print("Query-only vs Profile + Query (256)")
print(paired_query_vs_256.to_string(index=False))

print("\n" + "=" * 70 + "\n")

# Query-only vs 512-token profile-conditioned model
paired_query_vs_512 = paired_outcomes(
    query_only_retrievals,
    profile_query_512_retrievals,
    "Query-only",
    "Profile + Query (512)"
)

print("Query-only vs Profile + Query (512)")
print(paired_query_vs_512.to_string(index=False))

Query-only vs Profile + Query (256)
                             Outcome  Count  Percentage
                   Neither retrieves   5511   99.136535
Only Profile + Query (256) retrieves     33    0.593632
           Only Query-only retrieves     10    0.179888
                       Both retrieve      5    0.089944


Query-only vs Profile + Query (512)
                             Outcome  Count  Percentage
                   Neither retrieves   5507   99.064580
Only Profile + Query (512) retrieves     37    0.665587
           Only Query-only retrieves      9    0.161900
                       Both retrieve      6    0.107933


## 11.X Ground-Truth Rank Changes

For evaluation examples where both methods retrieve the ground-truth product within the Top-50, the ground-truth rank is compared between query-only retrieval and profile-conditioned retrieval.

A lower numerical rank indicates that the ground-truth product appears earlier in the retrieved list. The analysis identifies cases in which profile conditioning improves, worsens or leaves the ground-truth rank unchanged.

In [28]:
# Compare ground-truth ranks for examples retrieved by both methods

def compare_ranks(df_a, df_b, name_a, name_b):
    merged = df_a[
        ["sample_id", "ground_truth_asin", "retrieved_asins"]
    ].merge(
        df_b[
            ["sample_id", "retrieved_asins"]
        ],
        on="sample_id",
        suffixes=("_a", "_b")
    )

    rows = []

    for _, row in merged.iterrows():

        rank_a = get_ground_truth_rank(
            row["retrieved_asins_a"],
            row["ground_truth_asin"]
        )

        rank_b = get_ground_truth_rank(
            row["retrieved_asins_b"],
            row["ground_truth_asin"]
        )

        # Only compare examples where both methods retrieve
        if rank_a is not None and rank_b is not None:

            rows.append({
                "sample_id": row["sample_id"],
                f"{name_a} rank": rank_a,
                f"{name_b} rank": rank_b,
                "rank_change": rank_a - rank_b
            })

    rank_comparison = pd.DataFrame(rows)

    if len(rank_comparison) > 0:
        rank_comparison["direction"] = np.select(
            [
                rank_comparison["rank_change"] > 0,
                rank_comparison["rank_change"] < 0,
                rank_comparison["rank_change"] == 0
            ],
            [
                "Profile-conditioned rank improves",
                "Profile-conditioned rank worsens",
                "Unchanged"
            ],
            default="Unknown"
        )

    return rank_comparison


rank_comparison_256 = compare_ranks(
    query_only_retrievals,
    proposed_retrievals,
    "Query-only",
    "Profile + Query (256)"
)

rank_comparison_512 = compare_ranks(
    query_only_retrievals,
    profile_query_512_retrievals,
    "Query-only",
    "Profile + Query (512)"
)


print("Query-only vs Profile + Query (256)")
print("Examples retrieved by both:", len(rank_comparison_256))

if len(rank_comparison_256) > 0:
    print(
        rank_comparison_256["direction"]
        .value_counts()
        .to_string()
    )

    print("\nRank comparisons:")
    print(rank_comparison_256.to_string(index=False))


print("\n" + "=" * 70 + "\n")


print("Query-only vs Profile + Query (512)")
print("Examples retrieved by both:", len(rank_comparison_512))

if len(rank_comparison_512) > 0:
    print(
        rank_comparison_512["direction"]
        .value_counts()
        .to_string()
    )

    print("\nRank comparisons:")
    print(rank_comparison_512.to_string(index=False))

Query-only vs Profile + Query (256)
Examples retrieved by both: 5
direction
Profile-conditioned rank improves    3
Profile-conditioned rank worsens     2

Rank comparisons:
 sample_id  Query-only rank  Profile + Query (256) rank  rank_change                         direction
      2642               16                          26          -10  Profile-conditioned rank worsens
      2974               21                           7           14 Profile-conditioned rank improves
      2983               12                           1           11 Profile-conditioned rank improves
      3163               19                          42          -23  Profile-conditioned rank worsens
      4924               47                          25           22 Profile-conditioned rank improves


Query-only vs Profile + Query (512)
Examples retrieved by both: 6
direction
Profile-conditioned rank improves    3
Profile-conditioned rank worsens     2
Unchanged                            1

Rank comparis

## 11.X Ablation Evaluation

The ablation experiments investigate two design choices in the profile-conditioned retrieval approach: maximum input sequence length and the ordering of the user profile and current query.

The primary configuration uses the user preference profile followed by the synthetic query with a maximum sequence length of 256 tokens. Two variants are evaluated: a 512-token profile-first representation and a 512-token query-first representation.

All three configurations use the same E5-base-v2 encoder, full Amazon Electronics catalogue, FAISS index, Top-50 retrieval depth and 5,559 evaluation examples. Therefore, differences in retrieval performance can be examined as effects of the input representation rather than changes to the retrieval infrastructure.

In [29]:
# Dedicated ablation evaluation

ablation_conditions = {
    "Profile + Query (256)": proposed_retrievals,
    "Profile + Query (512)": profile_query_512_retrievals,
    "Query + Profile (512)": query_profile_512_retrievals,
}

ablation_results = []

for condition_name, retrievals in ablation_conditions.items():

    for k in K_VALUES:

        recall_values = []
        mrr_values = []
        ndcg_values = []

        for _, row in retrievals.iterrows():

            retrieved = row["retrieved_asins"]
            ground_truth = row["ground_truth_asin"]

            recall_values.append(
                recall_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

            mrr_values.append(
                mrr_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

            ndcg_values.append(
                ndcg_at_k(
                    retrieved,
                    ground_truth,
                    k
                )
            )

        ablation_results.append({
            "Condition": condition_name,
            "K": k,
            "Recall@K": np.mean(recall_values),
            "MRR@K": np.mean(mrr_values),
            "NDCG@K": np.mean(ndcg_values)
        })

ablation_results = pd.DataFrame(ablation_results)

print(ablation_results.to_string(index=False))

            Condition  K  Recall@K    MRR@K   NDCG@K
Profile + Query (256)  1  0.000899 0.000899 0.000899
Profile + Query (256)  5  0.002518 0.001526 0.001774
Profile + Query (256) 10  0.003778 0.001695 0.002182
Profile + Query (256) 20  0.004497 0.001746 0.002365
Profile + Query (256) 50  0.006836 0.001827 0.002840
Profile + Query (512)  1  0.000540 0.000540 0.000540
Profile + Query (512)  5  0.002518 0.001280 0.001589
Profile + Query (512) 10  0.003958 0.001475 0.002057
Profile + Query (512) 20  0.005037 0.001547 0.002327
Profile + Query (512) 50  0.007735 0.001634 0.002865
Query + Profile (512)  1  0.000720 0.000720 0.000720
Query + Profile (512)  5  0.001799 0.001022 0.001209
Query + Profile (512) 10  0.002878 0.001164 0.001556
Query + Profile (512) 20  0.003778 0.001215 0.001769
Query + Profile (512) 50  0.006296 0.001299 0.002275


In [30]:
# Ablation hit counts alongside retrieval metrics

ablation_hit_results = []

for condition_name, retrievals in ablation_conditions.items():

    for k in K_VALUES:

        hits = 0

        for _, row in retrievals.iterrows():

            rank = get_ground_truth_rank(
                row["retrieved_asins"],
                row["ground_truth_asin"]
            )

            if rank is not None and rank <= k:
                hits += 1

        metric_row = ablation_results[
            (ablation_results["Condition"] == condition_name)
            & (ablation_results["K"] == k)
        ].iloc[0]

        ablation_hit_results.append({
            "Condition": condition_name,
            "K": k,
            "Hits": hits,
            "Hit rate (%)": 100 * hits / len(retrievals),
            "Recall@K": metric_row["Recall@K"],
            "MRR@K": metric_row["MRR@K"],
            "NDCG@K": metric_row["NDCG@K"]
        })

ablation_hit_results = pd.DataFrame(
    ablation_hit_results
)

print(
    ablation_hit_results.to_string(index=False)
)

            Condition  K  Hits  Hit rate (%)  Recall@K    MRR@K   NDCG@K
Profile + Query (256)  1     5      0.089944  0.000899 0.000899 0.000899
Profile + Query (256)  5    14      0.251844  0.002518 0.001526 0.001774
Profile + Query (256) 10    21      0.377766  0.003778 0.001695 0.002182
Profile + Query (256) 20    25      0.449721  0.004497 0.001746 0.002365
Profile + Query (256) 50    38      0.683576  0.006836 0.001827 0.002840
Profile + Query (512)  1     3      0.053967  0.000540 0.000540 0.000540
Profile + Query (512)  5    14      0.251844  0.002518 0.001280 0.001589
Profile + Query (512) 10    22      0.395755  0.003958 0.001475 0.002057
Profile + Query (512) 20    28      0.503688  0.005037 0.001547 0.002327
Profile + Query (512) 50    43      0.773520  0.007735 0.001634 0.002865
Query + Profile (512)  1     4      0.071955  0.000720 0.000720 0.000720
Query + Profile (512)  5    10      0.179888  0.001799 0.001022 0.001209
Query + Profile (512) 10    16      0.287822  0.002

## 11.X Ablation Differences Relative to the Primary Configuration

The primary profile-conditioned configuration uses a 256-token maximum sequence length with the user preference profile placed before the current query.

The following analysis reports the absolute change in retrieval metrics produced by increasing the maximum sequence length to 512 tokens and by reversing the input order. Positive values indicate an increase relative to the primary configuration, while negative values indicate a decrease.

In [31]:
# Calculate metric differences relative to Profile + Query (256)

primary = ablation_results[
    ablation_results["Condition"] == "Profile + Query (256)"
].copy()

primary = primary.set_index("K")

ablation_difference_results = []

for condition in [
    "Profile + Query (512)",
    "Query + Profile (512)"
]:

    variant = ablation_results[
        ablation_results["Condition"] == condition
    ].copy()

    variant = variant.set_index("K")

    for k in K_VALUES:

        ablation_difference_results.append({
            "Variant": condition,
            "K": k,
            "Recall difference": (
                variant.loc[k, "Recall@K"]
                - primary.loc[k, "Recall@K"]
            ),
            "MRR difference": (
                variant.loc[k, "MRR@K"]
                - primary.loc[k, "MRR@K"]
            ),
            "NDCG difference": (
                variant.loc[k, "NDCG@K"]
                - primary.loc[k, "NDCG@K"]
            )
        })

ablation_differences = pd.DataFrame(
    ablation_difference_results
)

print(
    ablation_differences.to_string(index=False)
)

              Variant  K  Recall difference  MRR difference  NDCG difference
Profile + Query (512)  1          -0.000360       -0.000360        -0.000360
Profile + Query (512)  5           0.000000       -0.000246        -0.000184
Profile + Query (512) 10           0.000180       -0.000221        -0.000125
Profile + Query (512) 20           0.000540       -0.000199        -0.000038
Profile + Query (512) 50           0.000899       -0.000192         0.000024
Query + Profile (512)  1          -0.000180       -0.000180        -0.000180
Query + Profile (512)  5          -0.000720       -0.000504        -0.000564
Query + Profile (512) 10          -0.000899       -0.000531        -0.000626
Query + Profile (512) 20          -0.000720       -0.000531        -0.000597
Query + Profile (512) 50          -0.000540       -0.000528        -0.000565


## 11.X Synthetic Query and Ground-Truth Consistency

The evaluation ground truth is defined by the held-out product associated with each synthetic query. Since the queries were generated synthetically, their semantic alignment with the held-out products is examined as an additional validity check.

This analysis is used to identify potential cases where the generated query does not adequately describe the held-out product. Such cases are not removed from the primary evaluation, but they are considered when interpreting retrieval performance.

In [32]:
# Inspect query-ground-truth consistency using product metadata

CATALOGUE_PATH = (
    DATA_DIR
    / "generated"
    / "full_product_catalogue.parquet"
)

catalogue = pd.read_parquet(
    CATALOGUE_PATH,
    columns=["parent_asin", "title", "store", "product_text"]
)

print("Catalogue shape:", catalogue.shape)

# Attach ground-truth product information
ground_truth_products = (
    eval_data[
        ["sample_id", "synthetic_query", "ground_truth_asin"]
    ]
    .merge(
        catalogue,
        left_on="ground_truth_asin",
        right_on="parent_asin",
        how="left"
    )
)

print("Evaluation examples:", len(ground_truth_products))
print(
    "Missing ground-truth products:",
    ground_truth_products["title"].isna().sum()
)

ground_truth_products[
    [
        "sample_id",
        "synthetic_query",
        "ground_truth_asin",
        "title",
        "store"
    ]
].head(10)

Catalogue shape: (1610012, 4)
Evaluation examples: 5559
Missing ground-truth products: 0


,sample_id,synthetic_query,ground_truth_asin,title,store
0,0,Looking for a high-speed micro sd card for my ...,B007VKII6A,IDANCE CRAZY101 - Watt - Channel Recording Stu...,IDANCE
1,1,looking for a good quality internal hard drive...,B07454F4JH,Seagate Backup Plus Portable 5TB External Hard...,Seagate
2,2,looking for a good quality internal hard drive...,B00SMVLPIK,Panasonic FOLDZ On-Ear Stereo Headphones with ...,Panasonic
3,3,looking for a good quality battery pack for my...,B0179DDIQI,BienSound HW50 Stereo Folding Headsets Strong ...,Biensound
4,4,looking for good quality on-ear headphones wit...,B00X088DZI,Kastar 5-Pack HHR-P105 Type 31 Battery Replace...,Kastar
5,5,Looking for a high-quality monocular for bird ...,B074BCTX1Q,Iball 5.8GHz Wireless Magnetic Trailer Hitch R...,IBall Wireless Trailer Hitch Camera
6,6,looking for a high-quality monocular for camping,B01CKPDUHW,Cameron Sino Replacement 1250mAh GPS Navigator...,Cameron Sino
7,7,best tv antenna for rv with 4k support,B07SRDPZ9Z,Corpco 6 x AA Battery Holder with Standard sna...,Corporate Computer
8,8,looking for a high-quality TV antenna to impro...,B01N91OF9H,Corporate Computer 8 AA Battery Holder with Sn...,Corporate Computer
9,9,looking for another battery holder like corpco,B07LGS3RTV,MOTOKU Air Filter Tune-Up Kit for Stihl 028 02...,MOTOKU


In [33]:
# Diagnostic: lexical overlap between query and ground-truth title

import re

def normalise_tokens(text):
    text = str(text).lower()
    return set(
        re.findall(r"\b[a-z0-9]+\b", text)
    )

ground_truth_products["query_tokens"] = (
    ground_truth_products["synthetic_query"]
    .apply(normalise_tokens)
)

ground_truth_products["title_tokens"] = (
    ground_truth_products["title"]
    .apply(normalise_tokens)
)

ground_truth_products["lexical_overlap"] = (
    ground_truth_products.apply(
        lambda row: (
            len(row["query_tokens"] & row["title_tokens"])
            / max(len(row["query_tokens"]), 1)
        ),
        axis=1
    )
)

print(
    ground_truth_products["lexical_overlap"].describe()
)

print(
    "\nExamples with zero lexical overlap:",
    (
        ground_truth_products["lexical_overlap"] == 0
    ).sum()
)

print(
    "Percentage with zero lexical overlap:",
    100 * (
        ground_truth_products["lexical_overlap"] == 0
    ).mean()
)

count    5559.000000
mean        0.125947
std         0.150971
min         0.000000
25%         0.000000
50%         0.100000
75%         0.200000
max         1.000000
Name: lexical_overlap, dtype: float64

Examples with zero lexical overlap: 2278
Percentage with zero lexical overlap: 40.97859327217125


In [34]:
# Save final quantitative evaluation results

FINAL_RESULTS_PATH = (
    DATA_DIR
    / "generated"
    / "final_evaluation_results.parquet"
)

# Combine main metrics and ablation metrics
all_final_metrics = final_metrics.copy()

all_final_metrics.to_parquet(
    FINAL_RESULTS_PATH,
    index=False
)

print("Saved:", FINAL_RESULTS_PATH)
print(
    "File size:",
    round(FINAL_RESULTS_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

print("\nFinal results:")
print(all_final_metrics.to_string(index=False))

Saved: /users/adhp511/query-conditioned-recommender/data/generated/final_evaluation_results.parquet
File size: 0.0 MB

Final results:
            Condition  K  Recall@K    MRR@K   NDCG@K
           Query-only  1  0.000000 0.000000 0.000000
           Query-only  5  0.000180 0.000045 0.000077
           Query-only 10  0.000540 0.000097 0.000198
           Query-only 20  0.001619 0.000176 0.000476
           Query-only 50  0.002698 0.000208 0.000684
Profile + Query (256)  1  0.000899 0.000899 0.000899
Profile + Query (256)  5  0.002518 0.001526 0.001774
Profile + Query (256) 10  0.003778 0.001695 0.002182
Profile + Query (256) 20  0.004497 0.001746 0.002365
Profile + Query (256) 50  0.006836 0.001827 0.002840
Profile + Query (512)  1  0.000540 0.000540 0.000540
Profile + Query (512)  5  0.002518 0.001280 0.001589
Profile + Query (512) 10  0.003958 0.001475 0.002057
Profile + Query (512) 20  0.005037 0.001547 0.002327
Profile + Query (512) 50  0.007735 0.001634 0.002865
Query + Profile (5

In [35]:
# Save ablation results and synthetic ground-truth diagnostics

ABLATION_RESULTS_PATH = (
    DATA_DIR
    / "generated"
    / "final_ablation_results.parquet"
)

GROUND_TRUTH_DIAGNOSTICS_PATH = (
    DATA_DIR
    / "generated"
    / "ground_truth_quality_diagnostics.parquet"
)

# Save ablation metrics
ablation_results.to_parquet(
    ABLATION_RESULTS_PATH,
    index=False
)

# Save ground-truth/query diagnostic information
ground_truth_diagnostics = ground_truth_products[
    [
        "sample_id",
        "synthetic_query",
        "ground_truth_asin",
        "title",
        "store",
        "lexical_overlap"
    ]
].copy()

ground_truth_diagnostics.to_parquet(
    GROUND_TRUTH_DIAGNOSTICS_PATH,
    index=False
)

print("Ablation results saved:")
print(ABLATION_RESULTS_PATH)
print(
    "Size:",
    round(ABLATION_RESULTS_PATH.stat().st_size / 1024, 2),
    "KB"
)

print("\nGround-truth diagnostics saved:")
print(GROUND_TRUTH_DIAGNOSTICS_PATH)
print(
    "Size:",
    round(
        GROUND_TRUTH_DIAGNOSTICS_PATH.stat().st_size / 1024,
        2
    ),
    "KB"
)

print("\nRows:")
print("Ablation results:", len(ablation_results))
print("Ground-truth diagnostics:", len(ground_truth_diagnostics))

Ablation results saved:
/users/adhp511/query-conditioned-recommender/data/generated/final_ablation_results.parquet
Size: 3.74 KB

Ground-truth diagnostics saved:
/users/adhp511/query-conditioned-recommender/data/generated/ground_truth_quality_diagnostics.parquet
Size: 638.86 KB

Rows:
Ablation results: 15
Ground-truth diagnostics: 5559
